# Script Outline



## Prepare Workspace

#### Import Packages

In [1]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

#### File paths

In [2]:
# Define user
user = os.getlogin()

# Working directories
path_sp  = os.path.join('C:\\Users', 'jfontes', 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', 'jfontes', 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Set file paths
path_config = os.path.join(path_git, 'Pipeline', 'Python Code', 'Census', 'aa_config')
path_out    = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')

#### User Defined Functions/Objects

In [3]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Inputs for Importing ACS Data

#### Import ACS tables/variables mapping and FIPS mapping

In [4]:
## Import Variable Mapping
df_vars   = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'ACS')
df_inputs = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'Inputs_ACS'
                        , dtype = {'msa': object})

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
year_start = int(df_inputs['year_start'].values[0])
year_end   = int(df_inputs['year_end'  ].values[0])
sp_folder_out = df_inputs['sp_folder'].values[0]

# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set tables and variables to import
list_vars = ['NAME'] + df_vars['ID'].to_list()
tables = df_vars['Table'].unique()

# Set years
years_to_import = list(range(year_start, year_end+1))



## Import County FIPS mapping
df_fips = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx')
                        , sheet_name = 'FIPSmapping'
                        , dtype = {'State FIPS': object, 'County FIPS': object})

# Convert to dictionary
dict_fips = df_fips[
                (df_fips['State'].isin(df_inputs['states'].values)) 
                & (df_fips['County Name'].isin(df_inputs['counties'].values))
]
dict_fips = dict_fips[['State FIPS', 'County FIPS']]

dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()


# Set MSA
df_inputs['msa'] = df_inputs['msa'].astype("string")
msa_to_import = df_inputs['msa'].values
msa_to_import = ",".join(msa_to_import)



# view
print(dict_fips)
print(msa_to_import)
print(tables)
print(indicator_name)
print(year_start)
print(year_end)
df_vars.head()

{}
12420,14260,16740,17140,17460,19740,19820,26900,28140,33100,33460,34980,36740,38060,38300,38900,39300,39580,40060,40900,41180,41620,41700,41860,41940,42660,45300,49700
['B25095']
Housing_Cost
2009
2022


,ID,Table Name,Label,Label_clean,Indicator Name,Variable,Race_Ethnicity,Table,Table2,Label2,Include,notes
20728,B25095_001E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,Estimate!!Total:,Total,Housing_Cost,Total,All,B25095,NaN,B25095_Total,Yes,NaN
20729,B25095_002E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,"Estimate!!Total:!!Less than $10,000:","Total Less than $10,000",Housing_Cost,"Total Less than $10,000",All,B25095,NaN,"B25095_Total Less than $10,000",Yes,NaN
20730,B25095_003E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,"Estimate!!Total:!!Less than $10,000:!!Less tha...","Total Less than $10,000 Less than 20.0 percent",Housing_Cost,"Total Less than $10,000 Less than 20.0 percent",All,B25095,NaN,"B25095_Total Less than $10,000 Less than 20.0 ...",Yes,NaN
20731,B25095_004E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,"Estimate!!Total:!!Less than $10,000:!!20.0 to ...","Total Less than $10,000 20.0 to 24.9 percent",Housing_Cost,"Total Less than $10,000 20.0 to 24.9 percent",All,B25095,NaN,"B25095_Total Less than $10,000 20.0 to 24.9 pe...",Yes,NaN
20732,B25095_005E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,"Estimate!!Total:!!Less than $10,000:!!25.0 to ...","Total Less than $10,000 25.0 to 29.9 percent",Housing_Cost,"Total Less than $10,000 25.0 to 29.9 percent",All,B25095,NaN,"B25095_Total Less than $10,000 25.0 to 29.9 pe...",Yes,NaN


## Import Data

#### Create Census Tracts Table

In [5]:

# # initialize empty list to store data frames
# list_df_acs = []

# # only want one table
# df_table = df_vars[df_vars['Table'] == tables[0]]
# list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+15] for x in range(0, len(df_table['ID'].to_list()), 15)]

# variables = []
# for x in list_table_vars:
#     variables.append(",".join(x))
# variables


In [6]:

# initialize empty list to store data frames
list_df_acs = []

# only want one table
df_table = df_vars[df_vars['Table'] == tables[0]]
list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+15] for x in range(0, len(df_table['ID'].to_list()), 15)]

variables = []
for x in list_table_vars:
    variables.append(",".join(x))
variables

# pull all years and counties for each table
for year in tqdm(years_to_import):
    try:
        temp =  acs5_msa(api_Key     = api_key
                         , variables = variables[0]
                         , year      = year
                         , msa       = msa_to_import)
        
        temp = temp[['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']]
        list_df_acs.append(temp)
        
    except:
        pass

# combine all years and counties
df_acs_raw = pd.concat(list_df_acs)
df_acs_raw.head()

100%|██████████| 14/14 [00:12<00:00,  1.09it/s]


,NAME,metropolitan statistical area/micropolitan statistical area,Year
0,"Indianapolis-Carmel-Anderson, IN Metro Area",26900,2014
1,"Austin-Round Rock, TX Metro Area",12420,2014
2,"Boise City, ID Metro Area",14260,2014
3,"Charlotte-Concord-Gastonia, NC-SC Metro Area",16740,2014
4,"Cincinnati, OH-KY-IN Metro Area",17140,2014


#### Import ACS Data by Census Tracts

In [7]:
# initialize empty list to store data frames
# import multiple years and counties
list_df_acs = []

# iterate through each table (pulling all tables at once fails because the URL is too long - i think)
for table in tables:

    # keep track of tables being imported
    print(table)
    list_df_tables = []

    # only want one table
    df_table = df_vars[df_vars['Table'] == tables[0]]
    list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+15] for x in range(0, len(df_table['ID'].to_list()), 15)]
    
    list_variables = []
    for x in list_table_vars:
        list_variables.append(",".join(x))
    
    # pull all years and counties for each table
    for variables in tqdm(list_variables):
        for year in years_to_import:
            try:
                list_df_tables.append(
                    acs5_msa(api_Key     = api_key
                             , variables = variables
                             , year      = year
                             , msa       = msa_to_import)
                )
            except:
                pass

        # combine all years and counties
        df_temp = pd.concat(list_df_tables)

    # left join data onto key
    df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'left')

B25095


100%|██████████| 5/5 [01:03<00:00, 12.68s/it]


In [8]:
# Check years and counties
print(df_acs_raw['Year'].unique())
# assert df_acs_raw['Year'].unique().tolist() == years_to_import
# assert df_acs_raw['msa' ].unique().tolist() == msa_to_import


# view raw data
pd.set_option('display.max_columns', None)
print(df_acs_raw.shape)
print(df_acs_raw.Year.unique())
df_acs_raw.head(3)

[2014 2015 2016 2017 2018 2019 2020 2021 2022]
(1260, 76)
[2014 2015 2016 2017 2018 2019 2020 2021 2022]


,NAME,metropolitan statistical area/micropolitan statistical area,Year,B25095_001E,B25095_002E,B25095_003E,B25095_004E,B25095_005E,B25095_006E,B25095_007E,B25095_008E,B25095_009E,B25095_010E,B25095_011E,B25095_012E,B25095_013E,B25095_014E,B25095_015E,B25095_016E,B25095_017E,B25095_018E,B25095_019E,B25095_020E,B25095_021E,B25095_022E,B25095_023E,B25095_024E,B25095_025E,B25095_026E,B25095_027E,B25095_028E,B25095_029E,B25095_030E,B25095_031E,B25095_032E,B25095_033E,B25095_034E,B25095_035E,B25095_036E,B25095_037E,B25095_038E,B25095_039E,B25095_040E,B25095_041E,B25095_042E,B25095_043E,B25095_044E,B25095_045E,B25095_046E,B25095_047E,B25095_048E,B25095_049E,B25095_050E,B25095_051E,B25095_052E,B25095_053E,B25095_054E,B25095_055E,B25095_056E,B25095_057E,B25095_058E,B25095_059E,B25095_060E,B25095_061E,B25095_062E,B25095_063E,B25095_064E,B25095_065E,B25095_066E,B25095_067E,B25095_068E,B25095_069E,B25095_070E,B25095_071E,B25095_072E,B25095_073E
0,"Indianapolis-Carmel-Anderson, IN Metro Area",26900,2014,484053,13516,160,98,199,162,200,579,8669,3449,28390,3362,2606,2283,2447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Indianapolis-Carmel-Anderson, IN Metro Area",26900,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1519,3829,12344,0,57932,18062,6223,5333,5136,5159,7337,10682,0,65455,23565,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Indianapolis-Carmel-Anderson, IN Metro Area",26900,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9765,10230,8027,5418,4902,3548,0,99776,49737,21600,13419,7127,3646,2737,1510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

In [9]:
# make copy
df_acs = df_acs_raw.copy()
df_acs = df_acs.rename(columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})

# mapping
df_msa_map = df_acs[df_acs['Year'] == 2022][['NAME', 'MSA_ID']].drop_duplicates()

df_msa_map.head()

,NAME,MSA_ID
1120,"Austin-Round Rock-Georgetown, TX Metro Area",12420
1125,"Boise City, ID Metro Area",14260
1130,"Charlotte-Concord-Gastonia, NC-SC Metro Area",16740
1135,"Cincinnati, OH-KY-IN Metro Area",17140
1140,"Cleveland-Elyria, OH Metro Area",17460


In [10]:
# make copy
df_acs = df_acs_raw.copy()
df_acs = df_acs.rename(columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})

# mapping
df_msa_map = df_acs[df_acs['Year'] == 2022][['NAME', 'MSA_ID']].drop_duplicates().rename(columns = {'NAME':'MSA'})
df_acs = df_acs.merge(df_msa_map, on = 'MSA_ID')

# Melt data from wide to long
df_acs = pd.melt(df_acs
                  , id_vars = ['NAME', 'MSA', 'MSA_ID', 'Year']
                  , var_name = 'ID'
                  , value_name = 'Total'
                 )


# Convert imported values to numeric
df_acs['Total'] = df_acs['Total'].apply(pd.to_numeric)


# Merge label 2
df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity']], on = 'ID', how = 'left')


# Subset
df_acs = df_acs[['ID', 'Table Name', 'Label', 'MSA_ID', 'MSA', 
                   'Year', 'Variable', 'Race_Ethnicity', 'Total']]


# view
df_acs.head()

,ID,Table Name,Label,MSA_ID,MSA,Year,Variable,Race_Ethnicity,Total
0,B25095_001E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,Estimate!!Total:,26900,"Indianapolis-Carmel-Anderson, IN Metro Area",2014,Total,All,484053.0
1,B25095_001E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,Estimate!!Total:,26900,"Indianapolis-Carmel-Anderson, IN Metro Area",2014,Total,All,NaN
2,B25095_001E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,Estimate!!Total:,26900,"Indianapolis-Carmel-Anderson, IN Metro Area",2014,Total,All,NaN
3,B25095_001E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,Estimate!!Total:,26900,"Indianapolis-Carmel-Anderson, IN Metro Area",2014,Total,All,NaN
4,B25095_001E,HOUSEHOLD INCOME BY SELECTED MONTHLY OWNER COS...,Estimate!!Total:,26900,"Indianapolis-Carmel-Anderson, IN Metro Area",2014,Total,All,NaN


## Organize Exports

In [11]:
# Sort by census tract then by year then by race/ethnicity
df_acs['Race_Ethnicity_sort'] = pd.Categorical(df_acs['Race_Ethnicity'], ['All'
                                                                 , 'AMERICAN INDIAN AND ALASKA NATIVE ALONE'
                                                                 , 'ASIAN ALONE'
                                                                 , 'BLACK OR FRICAN AMERICAN ALONE'
                                                                 , 'HISPANIC OR LATINO'
                                                                 , 'NATIVE HAWAIIAN AND OTHER PACIFIC ISLANDER ALONE'
                                                                 , 'WHITE ALONE'
                                                                 , 'WHITE ALONE, NOT HISPANIC OR LATINO'
                                                                 , 'SOME OTHER RACE ALONE', 'TWO OR MORE RACES'])


# sort and then remove categorical field
df_acs = df_acs.sort_values(by = ['MSA', 'Year', 'Race_Ethnicity_sort'], ascending = True)
df_acs = df_acs.drop(['Race_Ethnicity_sort'], axis = 1)

In [12]:
# MSA
df_msa1 = df_acs.groupby(['MSA', 
                          'Variable','Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Total'].sum()
# df_msa1['Proportion'] = df_msa1['Total'] / df_msa1[df_msa1['Variable'] != 'Total'].groupby(['MSA', 'Year'])['Total'].transform('sum')


# # missing values represent a population of 0
# df_msa1['Proportion'] = df_msa1['Proportion'].fillna(1)

# view
df_msa1.head()

,MSA,Variable,Year,Race_Ethnicity,Total
0,"Austin-Round Rock-Georgetown, TX Metro Area",Total,2014,All,390412.0
1,"Austin-Round Rock-Georgetown, TX Metro Area","Total Less than $10,000",2014,All,10530.0
2,"Austin-Round Rock-Georgetown, TX Metro Area","Total Less than $10,000 Less than 20.0 percent",2014,All,113.0
3,"Austin-Round Rock-Georgetown, TX Metro Area","Total Less than $10,000 20.0 to 24.9 percent",2014,All,59.0
4,"Austin-Round Rock-Georgetown, TX Metro Area","Total Less than $10,000 25.0 to 29.9 percent",2014,All,130.0


In [13]:
## Dcasts

# MSA
df_msa2_pop = df_msa1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Total').reset_index()
# df_msa2_prop = df_msa1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
#                                    , columns = 'Variable'
#                                    , values = 'Proportion').reset_index()


# missing values represent a population of 0
df_msa2_pop  = df_msa2_pop .fillna(0)
# df_msa2_prop = df_msa2_prop.fillna(1)


# view
# df_msa2_prop.head(3)

#### Export

In [14]:
# import re

# pattern = r'Cleveland'
# match = df_msa2_prop['MSA'].str.match(pattern)

# df_msa2_prop[match]





In [15]:
# Set output name
name_output_long = ['ACS5 ', indicator_name, ' Long MSA.xlsx']
name_output_wide = ['ACS5 ', indicator_name, ' Wide MSA.xlsx']

name_output_long = "".join(name_output_long)
name_output_wide = "".join(name_output_wide)


In [18]:
# Export long
with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
    df_acs      .to_excel(writer, index = False, sheet_name = 'Full MSA')
    df_msa1     .to_excel(writer, index = False, sheet_name = 'MSA'     )


In [19]:
# Export wide
with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
    df_msa2_pop      .to_excel(writer, index = False, sheet_name = 'MSA Total')
    # df_msa2_prop     .to_excel(writer, index = False, sheet_name = 'MSA Prop' )
